### IMPORTAMOS LAS LIBRERÍAS NECESARIAS

In [26]:
import random
import time
import numpy as np
from IPython.display import clear_output

### DEFINICIÓN DEL ARBOL

In [5]:
class node:
    """
    Clase para representar los nodos del árbol.
    """

    def __init__(self, tablero, turno, g_p):
        #estructuras de navegación
        self.pad=None            # referencia al nodo padre
        self.hijos = []          # lista de referencias a los nodos hijos

        #datos contenida
        self.tablero = tablero   # tablero del juego específico en el nodo
        self.g_p = g_p           # 1: ganador, -1: perdedor, 0: sigue el juego
        self.uct = 6.0           # calificación asignada al tablero
        self.n = 0               # cantidad de veces que se ha visitado el nodo
        self.wins = 0            # cantidad de victorias del subárbol
        self.turno = turno       # turno del siguiente tiro

### DEFINICIÓN DE LA CLASE PARA JUGAR

In [ ]:
class hexapawn:
    def __init__(self):
        self.seed = int(time.time_ns())
        self.gen = random.Random(self.seed)

    #Regresa un número entero aleatorio
    def random_int(self):
        return self.gen.randint(1, 1000)
    
    #Coloca el tablero para un juego desde 0
    def poner_el_juego(self, n, turno=0):
        '''
        pone el juego desde el inicio, donde el tablero es siempre cuadrado, y la cantidad
        de piezas dependen del tamaño del tablero
        Entrada:
        int n: tamaño del tablero
        int turno=0: turno actual (preset a cero, o sea turno humano)
        
        Salida:
        salida: tablero
        '''
        primera_fila=[2 for i in range(n)]
        ultima_fila=[1 for i in range(n)]
        fila_medio=[0 for i in range(n)]

        tablero=[]
        tablero.append(primera_fila.copy())
        for i in range(n-2): tablero.append(fila_medio.copy())
        tablero.append(ultima_fila.copy())

        return tablero, 1 if turno==0 or turno==1 else 2
    
    #Imprime de manera bonita el estado del juego, junto con el turno
    def ver_tablero(self, tablero, turno=1):
        clear_output()
        count=0
        print("╔", end="")
        for i in range(len(tablero)*4-1):
            print("═", end="")
            count+=1
            if(count==100):
                return 
            
        print("╗   filas     turno de: ", "X" if turno==1 else "O")
        
        for i in range(len(tablero)):
            print("║", end="")

            for j in range(len(tablero[1])):
                print("   " if tablero[i][j]==0 else " X " if tablero[i][j]==1 else " O ", end="")
                if j!=len(tablero[1])-1: print("│", end="")
                
            print("║ ", i)
            
            if i!=len(tablero):
                print("║", end="")
                for j in range(len(tablero[1])):
                    for k in range(3):
                        print("-", end="")
                        
                    if j!=len(tablero[1])-1:
                        print("┼", end="")
                print("║")
        
        print("╚", end="")
        for i in range(len(tablero[1])*4-1):
            print("═", end="")

        print("╝\n ", end="")
        for i in range(len(tablero[1])):
            print(f" {i}  ", end="")
            
        print("\n\ncolumnas\n\n")
        
    #Cambia de turno para el siguiente
    def siguiente_turno(self, turno):
        return 2 if turno==2 else 1
    
    #Filas original, columnas original, filas terminal, columnas terminal
    #te dice si el tiro que quieres hacer es legal o no
    def tiro_legal(self, f_o, c_o, f_t, c_t, tablero):
        if(f_t<0 or f_t>len(tablero)-1 or c_t<0 or c_t>len(tablero)-1):
            return False
        
        if(tablero[f_o][c_o]==1):
            return (((f_t==f_o-1) and (c_o==c_t)) and (tablero[f_t][c_t]==0)) or (((f_t==f_o-1) and (c_o==c_t-1)) and (tablero[f_t][c_t]==2)) or (((f_t==f_o-1) and (c_o==c_t+1)) and (tablero[f_t][c_t]==2))
        
        elif(tablero[f_o][c_o]==2):
            return (((f_t==f_o+1) and (c_o==c_t)) and (tablero[f_t][c_t]==0)) or (((f_t==f_o+1) and (c_o==c_t-1)) and (tablero[f_t][c_t]==1)) or (((f_t==f_o+1) and (c_o==c_t+1)) and (tablero[f_t][c_t]==1));
        
        else:
            return False
    
    #recibe un tablero junto con el turno y da un tiro aleatorio, regresa el tablero con el
    #tiro regristrado y el siguiente turno en una tupla
    def tiro_random(self, tablero, turno):
        #posibles almacena un vector, de coordenadas de todas las fichas del jugador del que queremos hacer el tiro
        posibles=[]

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]==turno):
                    posibles.append([i, j])

        self.gen.shuffle(posibles)
        
        while posibles:
            direcciones=[-1,0,1]
            elegido=posibles[len(posibles)-1]
            self.gen.shuffle(direcciones)
            
            while direcciones:
                if(self.tiro_legal(elegido[0],
                                   elegido[1],
                                   elegido[0]+(-1 if turno==1 else 1),
                                   elegido[1]+direcciones[len(direcciones)-1],
                                   tablero)):
                    tablero[elegido[0]][elegido[1]]=0
                    tablero[elegido[0]-1 if turno==1 else elegido[0]+1][elegido[1]+direcciones[len(direcciones)-1]]=turno

                    return (tablero, 2 if turno==1 else 1)
                
                else:
                    direcciones.pop()
                
            
            posibles.pop()
        
        return tablero, 0
    
    def todos_los_tiros(self, tablero, turno):
        posibles=[]

        todos=[]
        tablero_copia=tablero.copy()

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]==turno):
                    posibles.append([i, j])

        self.gen.shuffle(posibles)

        while(posibles):
            direcciones=[-1,0,1]
            elegido=posibles[posibles.size()-1]
            self.gen.shuffle(direcciones)
            
            while(direcciones):
                if(self.tiro_legal(elegido[0],
                                   elegido[1],
                                   elegido[0]+(-1 if turno==1 else 1),
                                   elegido[1]+direcciones[len(direcciones)-1],
                                   tablero)):
                    
                    tablero[elegido[0]][elegido[1]]=0
                    
                    tablero[elegido[0]-1 if turno==1 else elegido[0]+1][elegido[1]+direcciones[len(direcciones)-1]]=turno
                    todos.append(tablero)
                    tablero=tablero_copia
                    direcciones.pop()
                
                else:
                    direcciones.pop()
                
            
            posibles.pop()

        return todos, turno

    #condicion 1 de ganar: llegar al otro lado, regresa -1 si nadie satisface esta 
    #condición, de lo contrario, regresa la ficha ganadora
    def ganar_1(self, tablero):
        for i in range(0, len(tablero), len(tablero)-1):
            for j in range(len(tablero[0])):
                if i==0:
                    if(tablero[i][j]==1):
                        return True
                else:
                    if(tablero[i][j]==2):
                        return True
        return False
    
    #condicion 2 de ganar: uno de los dos ya no tiene fichas, regresa -1 si nadie satisface, 
    #de lo contrario regresa la ficha ganadora
    
    #creo que aquí hay un problema, no siempre detecta cuando ya no pueden seguir los tiros
    #al menos creo que esa es la razón de un problema que surgió al generar los tiros aleatorios
    #que para terminar los tiros pide un tablero con finalizacion, pero no se detectó por parte de esta funcion
    #pero ese fue un solo caso particular, en los demás si sigue funcionando
    def ganar_2(self, tablero):
        conteo=[0, 0]
        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j]!=0):
                    conteo[tablero[i][j]-1]+=1
                    
        return conteo[0]==0 or conteo[1]==0

    #condicion 3 de ganar: ya nadie puede tirar. regresa 1 si esta condicion se cumple,
    #regresa 0 de lo contrario
    def ganar_3(self, tablero):
        fichas=[]

        for i in range(len(tablero)):
            for j in range(len(tablero[0])):
                if(tablero[i][j])!=0:
                    fichas.append([i, j])

        for i in range(len(fichas)):
            direcciones=[-1, 0, 1]
            for j in range(3):
                if(self.tiro_legal(fichas[i][0],
                                   fichas[i][1],
                                   fichas[i][0]+(-1 if tablero[fichas[i][0]][fichas[i][1]]==1 else 1),
                                   fichas[i][1]+direcciones[j],
                                   tablero)):
                    return False

        return True
    
    #verifica las condiciones de terminado del juego
    def condiciones_de_terminado(self, tablero):
        return self.ganar_1(tablero) or self.ganar_2(tablero) or self.ganar_3(tablero)
    
    #le das un tablero y turno, y termina el juego con tiros aleatorios. Regresa 
    #la ficha ganadora. Adicionalmente, puedes ver qué onda con el juego
    def terminar_juego(self, tablero, turno, verbose=0):
        n=0
        while(1):
            tablero, turno=self.tiro_random(tablero, turno)
            if(verbose):
                self.ver_tablero(tablero,turno)

            if(self.condiciones_de_terminado(tablero) or turno==0):
                if(verbose):
                    if(turno==1):
                        print("VICTORIA DE O")
                    else:
                        print("VICTORIA DE O")
                        
                return 2 if turno==1 else 1
            
        return 0
    
    #te da un juego contra humano dado un tamaño de tablero, no regresa nada, solo es
    #para entretenimiento
    def juego_random_contra_humano(self, tamano):
        print("JUEGO CONTRA HUMANO:")
        tablero, turno=self.poner_el_juego(tamano)
        
        seguir=1
        self.ver_tablero(tablero,turno)

        while(seguir==1):
            while(1):
                f_o=int(input("\nFilas(origen):"))
                c_o=int(input("\nColumnas(origen):"))
                f_t=int(input("\nFilas(terminal):"))
                c_t=int(input("\nColumnas(terminal):"))
                if(self.tiro_legal(f_o, 
                                   c_o,
                                   f_t,
                                   c_t,
                                   tablero)==True):
                    tablero[f_o][c_o]=0
                    tablero[f_t][c_t]=1
                    break
                else:
                    print("\nTiro no permitido, intenta de nuevo.")

            if(turno==1):
                turno=2
            else:
                turno=1

            self.ver_tablero(tablero,turno)

            if(self.condiciones_de_terminado(tablero)):
                print("\nVICTORIA DE:", "O\n\n\nQuieres seguir?:(1=si, 0=no)" if turno==1 else "X\n\n\nQuieres seguir?:(1=si, 0=no)")
                seguir=int(input())
                if(seguir!=0):
                    tablero, turno=self.poner_el_juego(tamano)
                    self.ver_tablero(tablero,turno)
                
                else:
                    print("\nGracias por jugar, suerte para la próxima! :)" if turno==1 else "\nGracias por jugar, mejoraré para ganarte la próxima! >:)")
            
            else:
                print("\ntiro de maquina:")
                tablero, turno=self.tiro_random(tablero,turno)
                self.ver_tablero(tablero,turno)
                if(self.condiciones_de_terminado(tablero)):
                    print("\nVICTORIA DE:", "O\n\n\nQuieres seguir?:(1=si, 0=no)" if turno==1 else "X\n\n\nQuieres seguir?:(1=si, 0=no)")
                    seguir=int(input())
                    if(seguir!=0):
                        tablero, turno=self.poner_el_juego(tamano)
                        self.ver_tablero(tablero,turno)
                    
                    else:
                        print("\nGracias por jugar, suerte para la próxima! :)" if turno==1 else "\nGracias por jugar, mejoraré para ganarte la próxima! >:)")
    
if __name__=="__main__":
    juego=hexapawn()

    juego.juego_random_contra_humano(3)

╔═══════════╗   filas     turno de:  X
║   │   │ O ║  0
║---┼---┼---║
║   │ O │ X ║  1
║---┼---┼---║
║   │ X │   ║  2
║---┼---┼---║
╚═══════════╝
  0   1   2  

columnas



VICTORIA DE: O


Quieres seguir?:(1=si, 0=no)

Gracias por jugar, suerte para la próxima! :)
